In [1]:
!pip install mne numpy

import mne
import numpy as np
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# PATHS
# This assumes you uploaded your 'data' folder to 'EEG_Project' in Drive
raw_data_dir = '/content/drive/MyDrive/EEG_Project/data/files/'
save_path = '/content/drive/MyDrive/EEG_Project/data/processed/'

os.makedirs(save_path, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 41.0 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
N_SUBJECTS = 10  # Change to 109 to run all your files
Fs = 160

# Session Split
TRAIN_RUNS = [3, 4, 5, 6, 7, 8]
TEST_RUNS  = [9, 10, 11, 12, 13, 14]

In [7]:
def get_file_path(subject_id, run_id):
    """
    Constructs the path based on your folder structure:
    .../files/S001/S001R03.edf
    """
    # Format subject as 3 digits (e.g., 1 -> '001')
    subj_str = f"S{subject_id:03d}"
    # Format run as 2 digits (e.g., 3 -> '03')
    run_str = f"R{run_id:02d}"

    # Construct filename: S001R03.edf
    fname = f"{subj_str}{run_str}.edf"

    # Full path: .../files/S001/S001R03.edf
    return os.path.join(raw_data_dir, subj_str, fname)

def process_subject(subject_id, runs):
    try:
        # 1. Load the first run to initialize
        first_run_path = get_file_path(subject_id, runs[0])

        if not os.path.exists(first_run_path):
            print(f"  [Missing] File not found: {first_run_path}")
            return None

        raw = mne.io.read_raw_edf(first_run_path, preload=True, verbose=False)

        # 2. Append the rest of the runs
        for run_id in runs[1:]:
            fpath = get_file_path(subject_id, run_id)
            if os.path.exists(fpath):
                raw.append(mne.io.read_raw_edf(fpath, preload=True, verbose=False))
            else:
                print(f"  [Warning] Run {run_id} missing for Subject {subject_id}")

        # 3. Standardize & Filter
        # (We skip 'standardize' helper because it tries to fetch online sometimes,
        # instead we set montage directly)

        # Clean up channel names by removing trailing dots
        channel_names = raw.info['ch_names']
        mapping = {}
        for ch_name in channel_names:
            cleaned_ch_name = ch_name.rstrip('.')
            if cleaned_ch_name != ch_name:
                mapping[ch_name] = cleaned_ch_name

        if mapping:
            raw.rename_channels(mapping)

        montage = mne.channels.make_standard_montage('standard_1005')
        raw.set_montage(montage, on_missing='ignore')

        raw.filter(1., 40., fir_design='firwin', skip_by_annotation='edge', verbose=False)

        # 4. Epoching (4s windows)
        epochs = mne.make_fixed_length_epochs(raw, duration=4.0, overlap=2.0, verbose=False)

        return epochs.get_data(copy=False)

    except Exception as e:
        print(f"  Subject {subject_id} Error: {e}")
        return None

In [8]:
print(f"Reading data from: {raw_data_dir}")
print(f"Processing {N_SUBJECTS} subjects...")

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

for subject in range(1, N_SUBJECTS + 1):
    print(f"Processing Subject {subject}...")

    # Train Session
    data_train = process_subject(subject, TRAIN_RUNS)
    if data_train is not None:
        X_train_list.append(data_train)
        y_train_list.append(np.full(len(data_train), subject - 1))

    # Test Session
    data_test = process_subject(subject, TEST_RUNS)
    if data_test is not None:
        X_test_list.append(data_test)
        y_test_list.append(np.full(len(data_test), subject - 1))

# Concatenate
if len(X_train_list) > 0:
    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)
    X_test = np.concatenate(X_test_list, axis=0)
    y_test = np.concatenate(y_test_list, axis=0)

    # Normalize
    print("Normalizing data...")
    mean, std = np.mean(X_train), np.std(X_train)
    X_train = (X_train - mean) / std
    X_test  = (X_test - mean) / std

    # Save
    print(f"Saving to {save_path}...")
    np.save(f'{save_path}X_train.npy', X_train)
    np.save(f'{save_path}y_train.npy', y_train)
    np.save(f'{save_path}X_test.npy', X_test)
    np.save(f'{save_path}y_test.npy', y_test)
    print("Done!")
elif not X_train_list and N_SUBJECTS > 0:
    print("No data was processed. Please check if your data files exist in the specified path.")
else:
    print("No data was processed. Check your Drive paths.")

Reading data from: /content/drive/MyDrive/EEG_Project/data/files/
Processing 10 subjects...
Processing Subject 1...
Using data from preloaded Raw for 374 events and 640 original time points ...
8 bad epochs dropped
Using data from preloaded Raw for 374 events and 640 original time points ...
8 bad epochs dropped
Processing Subject 2...
Using data from preloaded Raw for 368 events and 640 original time points ...
8 bad epochs dropped
Using data from preloaded Raw for 368 events and 640 original time points ...
8 bad epochs dropped
Processing Subject 3...
Using data from preloaded Raw for 374 events and 640 original time points ...
8 bad epochs dropped
Using data from preloaded Raw for 374 events and 640 original time points ...
8 bad epochs dropped
Processing Subject 4...
Using data from preloaded Raw for 368 events and 640 original time points ...
8 bad epochs dropped
Using data from preloaded Raw for 368 events and 640 original time points ...
8 bad epochs dropped
Processing Subject 5